# Лабораторная 2 (CPU-вариант): декодерный трансформер на `Tiny Shakespeare`

Решение показывает полный перенос авторегрессионного контура с синтетики на реальный корпус
в обязательном базовом режиме `CPU-friendly`.


## Маршрут решения

1. Детерминированная загрузка корпуса и построение словаря.
2. Формирование окон и индексное разбиение.
3. Сборка декодера с причинной маской.
4. Обучение и сравнение с частотным ориентиром.
5. Детерминированная генерация по фиксированным подсказкам.
6. Диагностика внимания без доступа в будущее.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _detect_notebook_platform():
    """Определяет тип среды выполнения текущей тетради.

    Args:
      Нет.

    Returns:
      Строка из множества `{'local', 'colab', 'kaggle'}`.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    """Проверяет, похож ли путь на корень учебного репозитория.

    Args:
      path: Проверяемый путь.

    Returns:
      `True`, если обнаружены ключевые признаки корня репозитория.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    """Возвращает стандартный путь клонирования для облачной платформы.

    Args:
      platform: Имя платформы (`'colab'` или `'kaggle'`).

    Returns:
      Абсолютный путь каталога репозитория.

    Raises:
      ValueError: Если передано неподдерживаемое имя платформы.
    """
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    """Проверяет, остался ли в настройке шаблонный URL репозитория.

    Args:
      repo_url: Проверяемый URL репозитория.

    Returns:
      `True`, если URL имеет вид шаблона-заглушки.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    """Ищет корень курса, поднимаясь от текущего каталога вверх.

    Args:
      Нет.

    Returns:
      Объект `Path` корня репозитория или `None`, если путь не найден.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    """Обеспечивает доступность модуля `course_runtime` для текущей среды.

    Args:
      runtime_mode: Режим запуска тетради.
      repo_url: URL репозитория курса для облачной автозагрузки.

    Returns:
      `None`.

    Raises:
      ModuleNotFoundError: Если локальный запуск выполнен вне корректного корня репозитория.
      RuntimeError: Если в облаке отсутствует валидный URL репозитория или каталог повреждён.
      subprocess.CalledProcessError: Если команда `git clone` завершается с ошибкой.
    """
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Константы CPU-варианта


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

SEED = 17
PAD_ID = 0
CHECK_GEN_STEPS = 12
PROMPT_COUNT = 20

cfg = {
    'chars': 250_000,
    'context': 64,
    'stride': 3,
    'batch_size': 64,
    'embed_dim': 96,
    'num_heads': 4,
    'ff_dim': 192,
    'epochs': 3,
    'learning_rate': 2e-3,
    'gen_match_ratio': 0.20,
    'gen_threshold': 3,
    'gen_mean_threshold': 0.10,
}

plt.style.use('default')
keras.utils.set_random_seed(SEED)


## Теоретический ориентир

Перплексия рассчитывается как `exp(loss)` и сравнивается с частотным базовым ориентиром,
построенным на обучающей части корпуса.


## Шаг 1: загрузка корпуса и построение словаря


In [ ]:
def load_tiny_shakespeare(profile_cfg):
    """Загружает корпус и строит детерминированное символьное кодирование.

    Args:
      profile_cfg: Словарь параметров выбранного профиля.

    Returns:
      Кортеж `(text, encoded_ids, vocab, char_to_id, id_to_char)`.

    Raises:
      ValueError: Если выбранный профиль задаёт слишком короткий фрагмент текста.
    """
    path = tf.keras.utils.get_file(
        'tiny_shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
    )
    full_text = Path(path).read_text(encoding='utf-8')
    text = full_text[: profile_cfg['chars']]

    min_len = profile_cfg['context'] + CHECK_GEN_STEPS + 2
    if len(text) < min_len:
        raise ValueError('Срез корпуса слишком короткий для выбранного профиля.')

    chars = sorted(set(text))
    vocab = ['<PAD>', *chars]
    char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
    id_to_char = {idx: ch for ch, idx in char_to_id.items()}

    encoded_ids = np.asarray([char_to_id[ch] for ch in text], dtype=np.int32)
    return text, encoded_ids, vocab, char_to_id, id_to_char


text, encoded_ids, vocab, char_to_id, id_to_char = load_tiny_shakespeare(cfg)
VOCAB_SIZE = len(vocab)
print('Режим выполнения: cpu_friendly')
print('Длина текста:', len(text))
print('Размер словаря:', VOCAB_SIZE)


In [ ]:
assert len(text) == cfg['chars']
assert encoded_ids.dtype == np.int32
assert vocab[PAD_ID] == '<PAD>'

text_b, ids_b, _, _, _ = load_tiny_shakespeare(cfg)
assert np.array_equal(encoded_ids[:1000], ids_b[:1000])
print('Детерминизм загрузки подтверждён.')


## Шаг 2: окна и индексное разбиение


In [ ]:
def build_windows(encoded_stream, context_len, stride):
    """Строит обучающие окна фиксированной длины.

    Args:
      encoded_stream: Одномерный массив кодов символов.
      context_len: Длина входного контекста.
      stride: Шаг между соседними окнами.

    Returns:
      Кортеж `(X, Y, starts)`.

    Raises:
      ValueError: Если поток слишком короткий для построения хотя бы одного окна.
    """
    max_start = len(encoded_stream) - (context_len + 1)
    if max_start < 0:
        raise ValueError('Недостаточно символов для построения окон.')

    starts = np.arange(0, max_start + 1, stride, dtype=np.int32)
    windows = np.stack([encoded_stream[s:s + context_len + 1] for s in starts], axis=0)
    X = windows[:, :-1]
    Y = windows[:, 1:]
    return X.astype(np.int32), Y.astype(np.int32), starts


X_all, y_all, starts_all = build_windows(encoded_ids, cfg['context'], cfg['stride'])

total = len(X_all)
train_end = int(total * 0.8)
val_end = int(total * 0.9)

X_train, y_train, starts_train = X_all[:train_end], y_all[:train_end], starts_all[:train_end]
X_val, y_val, starts_val = X_all[train_end:val_end], y_all[train_end:val_end], starts_all[train_end:val_end]
X_test, y_test, starts_test = X_all[val_end:], y_all[val_end:], starts_all[val_end:]

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)


In [ ]:
assert X_train.shape[1] == cfg['context']
assert y_train.shape == X_train.shape
assert starts_train.ndim == 1
assert np.array_equal(X_train[0, 1:], y_train[0, :-1])
print('Окон train/val/test:', len(X_train), len(X_val), len(X_test))


## Шаг 3: декодерный блок с причинной маской


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Args:
      seq_len: Длина последовательности.

    Returns:
      Булев тензор формы `(seq_len, seq_len)`.

    Raises:
      tf.errors.InvalidArgumentError: Если `seq_len` не является положительным.
    """
    seq_len = tf.cast(seq_len, tf.int32)
    tf.debugging.assert_positive(seq_len, message='seq_len должен быть положительным.')
    shape = tf.stack([seq_len, seq_len])
    return tf.linalg.band_part(tf.ones(shape, dtype=tf.bool), -1, 0)


class TokenAndPositionEmbedding(layers.Layer):
    """Складывает токенные и позиционные векторы.

    Args:
      maxlen: Максимальная длина контекста.
      vocab_size: Размер словаря.
      embed_dim: Размер скрытого представления.
      **kwargs: Дополнительные аргументы базового слоя.

    Returns:
      Экземпляр слоя встраивания.

    Raises:
      ValueError: Если `embed_dim` меньше 1.
    """

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует слой токенного и позиционного встраивания.

        Args:
          maxlen: Максимальная длина контекста.
          vocab_size: Размер словаря токенов.
          embed_dim: Размерность векторного представления.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Returns:
          None.

        Raises:
          ValueError: Если `embed_dim` меньше 1.
        """
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Суммирует токенные и позиционные векторы.

        Args:
          inputs: Матрица идентификаторов формы `(batch, time)`.

        Returns:
          Тензор формы `(batch, time, embed_dim)`.

        Raises:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        positions = tf.range(start=0, limit=tf.shape(inputs)[-1], delta=1)
        token_vectors = self.token_embedding(inputs)
        position_vectors = self.position_embedding(positions)
        return token_vectors + position_vectors

    def compute_mask(self, inputs, mask=None):
        """Пробрасывает маску непустых токенов.

        Args:
          inputs: Матрица токенов.
          mask: Входная маска.

        Returns:
          Булева маска формы `(batch, time)`.

        Raises:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской.

    Args:
      embed_dim: Размер скрытого представления.
      num_heads: Число голов внимания.
      ff_dim: Размер скрытого слоя позиционно-независимой сети.
      rate: Доля прореживания.
      **kwargs: Дополнительные аргументы базового слоя.

    Returns:
      Экземпляр декодерного блока.

    Raises:
      ValueError: Если `embed_dim` не делится на `num_heads`.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои декодерного блока.

        Args:
          embed_dim: Размерность входных признаков.
          num_heads: Число голов внимания.
          ff_dim: Размер скрытого слоя позициионно-независимой сети.
          rate: Доля выключаемых нейронов в прореживании.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Returns:
          None.

        Raises:
          ValueError: Если `embed_dim` не делится на `num_heads`.
        """
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через маскированное самовнимание и FFN.

        Args:
          inputs: Тензор формы `(batch, time, embed_dim)`.
          padding_mask: Булева маска формы `(batch, time)`.
          training: Признак режима обучения.
          return_attention_scores: Нужно ли вернуть веса внимания.

        Returns:
          Либо выходной тензор, либо кортеж `(выход, attention_scores)`.

        Raises:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        seq_len = tf.shape(inputs)[1]
        causal_mask = build_causal_mask(seq_len)[tf.newaxis, :, :]

        if padding_mask is None:
            attention_mask = causal_mask
        else:
            # Ключи и запросы одновременно ограничиваются непустыми позициями.
            key_mask = tf.cast(padding_mask[:, tf.newaxis, :], tf.bool)
            query_mask = tf.cast(padding_mask[:, :, tf.newaxis], tf.bool)
            attention_mask = causal_mask & key_mask & query_mask

        attention_output, attention_scores = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            return_attention_scores=True,
            training=training,
        )

        x = self.norm_1(inputs + self.dropout_1(attention_output, training=training))
        y = self.ffn(x)
        output = self.norm_2(x + self.dropout_2(y, training=training))

        if return_attention_scores:
            return output, attention_scores
        return output

    def compute_mask(self, inputs, mask=None):
        """Отключает автоматическое пробрасывание маски в выход блока.

        Args:
          inputs: Входной тензор признаков.
          mask: Маска, пришедшая от предыдущего слоя.

        Returns:
          `None`, чтобы маскирование контролировалось явно в функции потерь.

        Raises:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        return None


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию по непустым позициям.

    Args:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Returns:
      Среднее значение функции потерь.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(per_token * mask) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность по непустым позициям.

    Args:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Returns:
      Доля верных токенов.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует значение функции потерь в перплексию.

    Args:
      loss_value: Средняя перекрёстная энтропия.

    Returns:
      Значение перплексии.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return float(np.exp(loss_value))


def frequency_baseline_perplexity(y_train_data, y_test_data, vocab_size, pad_id=PAD_ID):
    """Считает частотный базовый ориентир перплексии.

    Args:
      y_train_data: Целевые токены обучающей выборки.
      y_test_data: Целевые токены тестовой выборки.
      vocab_size: Размер словаря.
      pad_id: Идентификатор дополнения.

    Returns:
      Значение базовой перплексии.

    Raises:
      ValueError: Если в данных нет полезных токенов.
    """
    train_tokens = y_train_data[y_train_data != pad_id].reshape(-1)
    test_tokens = y_test_data[y_test_data != pad_id].reshape(-1)
    if len(train_tokens) == 0 or len(test_tokens) == 0:
        raise ValueError('Для базового ориентира нужны непустые токены.')

    counts = np.bincount(train_tokens, minlength=vocab_size).astype(np.float64)
    probs = counts / counts.sum()
    probs = np.maximum(probs, 1e-12)

    nll = -np.mean(np.log(probs[test_tokens]))
    return float(np.exp(nll))


## Шаг 4: сборка и обучение модели


In [ ]:
keras.utils.set_random_seed(SEED)

token_ids = keras.Input(shape=(cfg['context'],), dtype='int32', name='token_ids')
padding_mask = layers.Lambda(lambda t: tf.not_equal(t, PAD_ID), name='padding_mask')(token_ids)

embedding_layer = TokenAndPositionEmbedding(
    cfg['context'],
    VOCAB_SIZE,
    cfg['embed_dim'],
    name='token_position_embedding',
)
decoder_layer = CausalDecoderBlock(
    cfg['embed_dim'],
    cfg['num_heads'],
    cfg['ff_dim'],
    rate=0.1,
    name='causal_decoder',
)

x = embedding_layer(token_ids)
x = decoder_layer(x, padding_mask=padding_mask)
y_pred = layers.Dense(VOCAB_SIZE, activation='softmax', name='next_token_distribution')(x)

model = keras.Model(token_ids, y_pred, name='tiny_shakespeare_decoder')
model.compile(
    optimizer=keras.optimizers.Adam(cfg['learning_rate']),
    loss=masked_sparse_crossentropy,
    metrics=[masked_token_accuracy],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg['epochs'],
    verbose=2,
)

test_loss, test_token_accuracy = model.evaluate(test_ds, verbose=0)
test_perplexity = perplexity_from_loss(test_loss)
baseline_perplexity = frequency_baseline_perplexity(y_train, y_test, VOCAB_SIZE)

print(f"test_loss            : {test_loss:.4f}")
print(f"test_token_accuracy  : {test_token_accuracy:.4f}")
print(f"test_perplexity      : {test_perplexity:.4f}")
print(f"baseline_perplexity  : {baseline_perplexity:.4f}")

assert test_perplexity < baseline_perplexity, 'Модель должна превосходить частотный ориентир.'


In [ ]:
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in history.history['loss']]
val_ppl = [perplexity_from_loss(v) for v in history.history['val_loss']]
plt.plot(train_ppl, label='train_ppl')
plt.plot(val_ppl, label='val_ppl')
plt.xlabel('epoch')
plt.ylabel('perplexity')
plt.legend()
plt.tight_layout()


## Шаг 5: детерминированная генерация


In [ ]:
def ids_to_text(token_ids, id_to_char_map):
    """Преобразует идентификаторы в строку.

    Args:
      token_ids: Последовательность идентификаторов.
      id_to_char_map: Отображение id -> символ.

    Returns:
      Строка символов.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return ''.join(id_to_char_map.get(int(token), '') for token in token_ids if int(token) != PAD_ID)


def generate_greedy(model, prompt_ids, steps, context_len):
    """Генерирует продолжение в режиме argmax.

    Args:
      model: Обученная модель.
      prompt_ids: Начальная последовательность идентификаторов.
      steps: Число новых токенов.
      context_len: Длина модельного контекста.

    Returns:
      Список токенов продолжения длины не больше `steps`.

    Raises:
      ValueError: Если подсказка пуста.
    """
    if len(prompt_ids) == 0:
        raise ValueError('Подсказка не может быть пустой.')

    generated = []
    history = list(prompt_ids)
    for _ in range(steps):
        model_input = np.full((1, context_len), PAD_ID, dtype=np.int32)
        clipped = history[-context_len:]
        model_input[0, -len(clipped):] = clipped

        probs = model.predict(model_input, verbose=0)[0]
        next_token = int(np.argmax(probs[-1]))
        generated.append(next_token)
        history.append(next_token)
    return generated


def build_prompt_targets(encoded_stream, start_indices, context_len, continuation_len, n_prompts):
    """Готовит фиксированные подсказки и эталонные продолжения из тестового фрагмента.

    Args:
      encoded_stream: Полный поток кодов корпуса.
      start_indices: Стартовые индексы окон тестовой части.
      context_len: Длина контекста модели.
      continuation_len: Длина целевого продолжения.
      n_prompts: Количество подсказок.

    Returns:
      Список пар `(prompt_ids, target_ids)`.

    Raises:
      ValueError: Если тестовых окон недостаточно.
    """
    valid_starts = [
        int(start) for start in start_indices
        if int(start) + context_len + continuation_len <= len(encoded_stream)
    ]
    if len(valid_starts) < n_prompts:
        raise ValueError('Недостаточно тестовых окон для фиксированного набора подсказок.')

    selected = np.linspace(0, len(valid_starts) - 1, n_prompts, dtype=int)
    pairs = []
    for idx in selected:
        start = valid_starts[int(idx)]
        prompt = encoded_stream[start:start + context_len]
        target = encoded_stream[start + context_len:start + context_len + continuation_len]
        pairs.append((prompt.tolist(), target.tolist()))
    return pairs


fixed_pairs = build_prompt_targets(
    encoded_stream=encoded_ids,
    start_indices=starts_test,
    context_len=cfg['context'],
    continuation_len=CHECK_GEN_STEPS,
    n_prompts=PROMPT_COUNT,
)

success_count = 0
match_ratios = []
for prompt_ids, target_ids in fixed_pairs:
    generated_ids = generate_greedy(model, prompt_ids, CHECK_GEN_STEPS, cfg['context'])
    match_ratio = float(np.mean(np.equal(generated_ids, target_ids)))
    match_ratios.append(match_ratio)
    if match_ratio >= cfg['gen_match_ratio']:
        success_count += 1

mean_match_ratio = float(np.mean(match_ratios)) if match_ratios else 0.0
print(
    f'Корректных продолжений: {success_count} из {len(fixed_pairs)} '
    f'(порог на подсказку: {cfg["gen_match_ratio"]:.2f})'
)
print(f'Средняя доля совпадений: {mean_match_ratio:.4f}')
assert success_count >= cfg['gen_threshold'], 'Порог по числу корректных продолжений не достигнут.'
assert mean_match_ratio >= cfg['gen_mean_threshold'], 'Средняя доля совпадений ниже целевого порога.'

for prompt_ids, target_ids in fixed_pairs[:3]:
    generated_ids = generate_greedy(model, prompt_ids, CHECK_GEN_STEPS, cfg['context'])
    print('prompt   :', ids_to_text(prompt_ids[-40:], id_to_char))
    print('expected :', ids_to_text(target_ids, id_to_char))
    print('generated:', ids_to_text(generated_ids, id_to_char))
    print('---')


## Шаг 6: диагностика внимания


In [ ]:
trace_inputs = keras.Input(shape=(cfg['context'],), dtype='int32', name='trace_tokens')
trace_mask = layers.Lambda(lambda t: tf.not_equal(t, PAD_ID), name='trace_mask')(trace_inputs)
trace_embeddings = embedding_layer(trace_inputs)
_, trace_attention_scores = decoder_layer(
    trace_embeddings,
    padding_mask=trace_mask,
    return_attention_scores=True,
)
trace_model = keras.Model(trace_inputs, trace_attention_scores, name='trace_model')

sample_tokens = X_test[:1]
attention_scores = trace_model.predict(sample_tokens, verbose=0)
mean_scores = attention_scores[0].mean(axis=0)
useful_len = cfg['context']
heatmap = mean_scores[:useful_len, :useful_len]

future_mass = float(np.triu(heatmap, k=1).sum())
total_mass = float(heatmap.sum())
future_ratio = future_mass / total_mass if total_mass > 0 else 0.0

print(f'Отношение массы внимания в будущем: {future_ratio:.8f}')
assert future_ratio < 1e-4, 'Обнаружено заметное внимание к будущим позициям.'

plt.figure(figsize=(6, 5))
plt.imshow(heatmap, cmap='magma')
plt.colorbar()
plt.title('Средняя карта внимания (Tiny Shakespeare)')
plt.xlabel('Позиция ключа')
plt.ylabel('Позиция запроса')
plt.tight_layout()


## Проверка завершения (CPU-вариант)

1. `test_perplexity < baseline_perplexity`.
2. Порог генерации достигнут.
3. Средняя доля совпадений достигнута.
4. Диагностика внимания показывает отсутствие доступа в будущее.
5. Значение `test_perplexity` сохранено как ориентир для `GPU`-варианта.
